In [0]:
flight_time_raw_df=spark.read.table('dev.spark_db.flight_time_raw')

In [0]:
display(flight_time_raw_df.limit(3))

#### 2. develop logi to transform CRS_DEP_Time to an interverl

In [0]:
from pyspark.sql.functions import *
step_1=(
    flight_time_raw_df.withColumns({
    "CRS_DEP_TIME_HH": expr("left(lpad(CRS_DEP_TIME,4,'0'),2)"),
    "CRS_DEP_TIME_MM": expr("right(lpad(CRS_DEP_TIME,4,'0'),2")
})
)

step_2 =(
    step_1.withColumn('CRS_DEP_TIME_NEW',expr("cast(concat(CRS_DEP_TIME_HH':'CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE"))
)

In [0]:
# from pyspark.sql.functions import *

# step_2 = (
#     flight_time_raw_df
#     .withColumn("CRS_DEP_TIME_HH", expr("left(lpad(CRS_DEP_TIME,4,'0'),2)"))
#     .withColumn("CRS_DEP_TIME_MM", expr("right(lpad(CRS_DEP_TIME,4,'0'),2)"))
#     .withColumn(
#         "CRS_DEP_TIME_NEW",
#         expr("CAST(concat(CRS_DEP_TIME_HH, ':', CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE)")
#     )
# )

In [0]:
def time_intervel(hh_value):
    from pyspark.sql.functions import expr
    result = expr(f"""cast(concat(left(lpad({hh_value},4,'0'),2),
                   ':',
                    right(lpad({hh_value},4,'0'),2))
                    AS INTERVAL HOUR TO MINUTE)""")
    
    return(result)

In [0]:
from pyspark.sql.functions import expr
result_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME": time_intervel("CRS_DEP_TIME"),
        "DEP_TIME": time_intervel("DEP_TIME"),
        "WHEELS_ON": time_intervel("WHEELS_ON"),
        "CRS_ARR_TIME": time_intervel("CRS_ARR_TIME"),
        "ARR_TIME": time_intervel("ARR_TIME"),
        "TAXI_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)")
    })
)

#### save the result into the table

In [0]:
result_df.limit(3).display()

In [0]:
result_df.write.mode("overwrite").saveAsTable('dev.spark_db.flight_time_raw')

In [0]:
from pyspark.sql.functions import col

df_1 = df.withColumn('minutes_intervel', col('CRS_ARR_TIME')*60 )

In [0]:
df_1.display()

In [0]:
%sql
select * from dev.spark_db.flight_time_raw
limit 3